# Text generation with a GPT model

*Corrected version (Chapter 19). The original is kept as `training_gpt2_original.ipynb`.*

The notebook trains a decoder-only Transformer (GPT-2 architecture) as a language model on news text. With a GPU you can set `SMALL = False` for the 124M-parameter GPT-2 configuration; the default is a small configuration that trains in a few minutes on a CPU.

In [ ]:
import os, math, random
import numpy as np
import torch, torch.nn as nn

SEED = 123
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SMALL = True          # False: GPT-2 small (124M parameters), needs a GPU

## Data

`df_file.csv` (BBC news) is used if present, else a Reuters subset. **Documents are shuffled and split into training and validation documents.** In the original the texts were joined in file order and the last 10 % of the characters were the validation set; since the file is sorted by label, the validation set consisted of a single topic and the validation loss did not measure generalisation to the whole data.

In [ ]:
if os.path.exists('df_file.csv'):
    import pandas as pd
    docs = pd.read_csv('df_file.csv')['Text'].tolist()
else:
    import nltk
    nltk.download('reuters', quiet=True)
    from nltk.corpus import reuters
    docs = [reuters.raw(f) for f in reuters.fileids()[:3000]]
docs = [" ".join(d.split()) for d in docs]             # remove newlines and repeated spaces
random.Random(SEED).shuffle(docs)
n_val = len(docs) // 10
train_docs, val_docs = docs[n_val:], docs[:n_val]
print(len(train_docs), 'training documents,', len(val_docs), 'validation documents')

## Tokenisation

GPT-2 uses byte-level BPE (`tiktoken`, 50 257 tokens). If `tiktoken` or its vocabulary file cannot be downloaded, a byte tokenizer (256 byte values + one end-of-text token) is used, so that everything below still runs. The end-of-text token separates the documents.

In [ ]:
try:
    import tiktoken
    enc = tiktoken.get_encoding('gpt2')
    encode = lambda s: enc.encode(s)
    decode = lambda ids: enc.decode(ids)
    EOT, VOCAB = enc.eot_token, enc.n_vocab
    print('using tiktoken gpt2, vocabulary', VOCAB)
except Exception as e:
    print('tiktoken not available (%s); using a byte tokenizer' % type(e).__name__)
    encode = lambda s: list(s.encode('utf-8'))
    decode = lambda ids: bytes(i for i in ids if i < 256).decode('utf-8', errors='replace')
    EOT, VOCAB = 256, 257

def to_ids(doc_list):
    out = []
    for d in doc_list:
        out += encode(d) + [EOT]
    return torch.tensor(out)

train_ids, val_ids = to_ids(train_docs), to_ids(val_docs)
print('tokens: train', len(train_ids), ' validation', len(val_ids))
print(encode('Hello, do you like tea?')[:10], '...')

# tokens are not words: compare the number of tokens per word in English and in Uzbek
en = "Yesterday the committee approved the budget for the new school."
uz = "Kecha qo'mita yangi maktab uchun byudjetni tasdiqladi."
for name, s in (('English', en), ('Uzbek', uz)):
    print(f'{name}: {len(s.split())} words, {len(encode(s))} tokens')
# The original Uzbek example concatenated two string literals without a space ("vatan" "Biz"),
# so the text it printed had a missing space ("vatanBiz").

## Model

Pre-LN GPT block: LayerNorm, causal multi-head attention, residual; LayerNorm, feed-forward (GELU, width $4d$), residual. The output head shares its weights with the token embedding (**weight tying**), which is what makes GPT-2 small 124M and not 163M parameters. The original only *subtracted* the head from the count; here the weights are really shared.

In [ ]:
if SMALL:
    CFG = dict(vocab_size=VOCAB, context_length=128, emb_dim=128, n_heads=4, n_layers=4, drop_rate=0.1)
else:
    CFG = dict(vocab_size=VOCAB, context_length=1024, emb_dim=768, n_heads=12, n_layers=12, drop_rate=0.1)

class MultiHeadAttention(nn.Module):
    def __init__(self, d, ctx, heads, drop):
        super().__init__()
        assert d % heads == 0
        self.h, self.dh = heads, d // heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.out = nn.Linear(d, d)
        self.drop = nn.Dropout(drop)
        self.register_buffer('mask', torch.triu(torch.ones(ctx, ctx), diagonal=1).bool())

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, T, self.h, self.dh).transpose(1, 2) for t in (q, k, v)]
        s = (q @ k.transpose(-2, -1)) / math.sqrt(self.dh)
        s = s.masked_fill(self.mask[:T, :T], float('-inf'))
        a = self.drop(torch.softmax(s, dim=-1))
        return self.out((a @ v).transpose(1, 2).reshape(B, T, D))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c['emb_dim']
        self.att = MultiHeadAttention(d, c['context_length'], c['n_heads'], c['drop_rate'])
        self.ff = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.n1, self.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.drop = nn.Dropout(c['drop_rate'])

    def forward(self, x):
        x = x + self.drop(self.att(self.n1(x)))
        return x + self.drop(self.ff(self.n2(x)))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        d = c['emb_dim']
        self.tok, self.pos = nn.Embedding(c['vocab_size'], d), nn.Embedding(c['context_length'], d)
        self.drop = nn.Dropout(c['drop_rate'])
        self.blocks = nn.Sequential(*[Block(c) for _ in range(c['n_layers'])])
        self.ln = nn.LayerNorm(d)
        self.head = nn.Linear(d, c['vocab_size'], bias=False)
        self.head.weight = self.tok.weight                 # weight tying
        nn.init.normal_(self.tok.weight, std=0.02); nn.init.normal_(self.pos.weight, std=0.02)

    def forward(self, idx):
        T = idx.shape[1]
        x = self.drop(self.tok(idx) + self.pos(torch.arange(T, device=idx.device)))
        return self.head(self.ln(self.blocks(x)))

model = GPT(CFG)
print('parameters (shared weights counted once):', f"{sum(p.numel() for p in model.parameters()):,}")

## Training

Random windows of the token stream are used as batches; the target is the input shifted by one token. The loss is the cross-entropy of the next token; its exponential is the *perplexity*. A model that guesses uniformly has loss $\ln V$.

In [ ]:
def pick_device():
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device='cuda'); return torch.device('cuda')
        except Exception:
            pass
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = pick_device(); print('device:', device)
model.to(device)
T, B = CFG['context_length'], (16 if SMALL else 4)

def get_batch(ids):
    ix = torch.randint(len(ids) - T - 1, (B,))
    x = torch.stack([ids[i:i + T] for i in ix]); y = torch.stack([ids[i + 1:i + T + 1] for i in ix])
    return x.to(device), y.to(device)

@torch.no_grad()
def estimate(ids, n=20):
    model.eval()
    L = []
    for _ in range(n):
        x, y = get_batch(ids)
        L.append(nn.functional.cross_entropy(model(x).flatten(0, 1), y.flatten()).item())
    model.train()
    return float(np.mean(L))

print('uniform guess: loss', round(math.log(VOCAB), 3))
STEPS = 400 if SMALL else 2000
opt = torch.optim.AdamW(model.parameters(), lr=1e-3 if SMALL else 4e-4, weight_decay=0.1)
for step in range(STEPS + 1):
    if step % 100 == 0:
        print(f'step {step:5d}  train {estimate(train_ids):.3f}  validation {estimate(val_ids):.3f}')
    x, y = get_batch(train_ids)
    loss = nn.functional.cross_entropy(model(x).flatten(0, 1), y.flatten())
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

## Generation

Greedy decoding always takes the most probable token and tends to repeat itself. *Temperature* $\tau$ divides the logits (small $\tau$ = more conservative, large $\tau$ = more random), and *top-$k$* keeps only the $k$ most probable tokens before sampling.

In [ ]:
@torch.no_grad()
def generate(prompt, n=120, temperature=1.0, top_k=None, greedy=False):
    model.eval()
    idx = torch.tensor([encode(prompt)], device=device)
    for _ in range(n):
        logits = model(idx[:, -T:])[0, -1]
        if greedy:
            nxt = logits.argmax().view(1, 1)
        else:
            logits = logits / temperature
            if top_k:
                logits[logits < logits.topk(top_k).values[-1]] = float('-inf')
            nxt = torch.multinomial(torch.softmax(logits, -1), 1).view(1, 1)
        idx = torch.cat([idx, nxt], dim=1)
    return decode(idx[0].tolist())

prompt = 'The company said '
for name, kw in (('greedy', dict(greedy=True)), ('temperature 0.7', dict(temperature=0.7)),
                 ('temperature 1.5', dict(temperature=1.5)), ('top-k 10', dict(top_k=10))):
    print('---', name); print(generate(prompt, **kw))

## Saving the model

In [ ]:
torch.save({'cfg': CFG, 'state_dict': model.state_dict()}, 'gpt_small.pt')